In [7]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from pathlib import Path
import helper
import seaborn as sns

# Increase default font sizes for readability
mpl.rcParams.update({
    'font.size': 28,
    'axes.titlesize': 30,
    'axes.labelsize': 28,
    'xtick.labelsize': 26,
    'ytick.labelsize': 26,
    'legend.fontsize': 28,
    'legend.title_fontsize': 28,
})

# Set style params
FIGSIZE_DOUBLE = (20, 10)
SAVE_KWARGS = {'bbox_inches': 'tight', 'dpi': 300}

# Ensure plots_output exists
OUTPUT_DIR = Path("plots_output")
OUTPUT_DIR.mkdir(exist_ok=True)


In [8]:
# Paths
# FICE times are in the current repo
FICE_TIMES_PATH = Path("../scripts/inference_times.csv")

# GCARE runs are in the sibling FICE repo
GCARE_PATH = Path("/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/notebooks/gcare_runs")

# Constants
GRAPHS = [
    'swdf',
    'dbpedia100k',
    'fb237_v4',
    'lubm',
    'wikidata',
    'wn18rr_v4',
]

# Mapping: Graph -> (GCARE Folder Prefix, FICE Folder Name, FICE Dataset Name)
CONFIG_MAP = {
    'swdf': ('swdf', 'swdf', 'swdf'),
    'dbpedia100k': ('dbpedia100k', 'dbpedia100k', 'dbpedia100k'),
    'fb237_v4': ('fb237_v4_ind', 'fb237', 'fb237_v4_ind'),
    'lubm': ('lubm', 'lubm', 'lubm'),
    'wikidata': ('wikidata', 'wikidata', 'wikidata'),
    'wn18rr_v4': ('wn18rr_v4_ind', 'wn18rr', 'wn18rr_v4_ind'),
}

In [ ]:
def plot_time_boxplot_per_cardinality(df: pd.DataFrame, output_path: Path):
    if df.empty:
        return

    # Bins for cardinality
    bins = [0, 1, 10, 100, 1000, np.inf]
    labels = ['0-1', '2-10', '11-100', '101-1000', '>1000']

    df = df.copy()
    df['bin'] = pd.cut(df['true_value'], bins=bins, labels=labels, include_lowest=True)
    df = df.dropna(subset=['bin'])

    if df.empty:
        return

    try:
        # Determine hue order (preferred order first)
        actual_methods = set(df['method'].unique())
        preferred_order = ['fice', 'cset', 'wj']
        hue_order = [m for m in preferred_order if m in actual_methods] + [m for m in actual_methods if m not in preferred_order]

        # Palette mapping
        palette = {'fice': 'tab:blue', 'cset': 'tab:orange', 'wj': 'tab:green'}

        # Create grouped boxplot using seaborn.catplot (dodge groups by hue)
        g = sns.catplot(kind='box', data=df, x='bin', y='time', hue='method',
                        palette=palette, hue_order=hue_order, order=labels,
                        height=FIGSIZE_DOUBLE[1], aspect=FIGSIZE_DOUBLE[0]/FIGSIZE_DOUBLE[1],
                        dodge=True, showfliers=False, linewidth=1.5)
        ax = g.ax

        ax.set_yscale('log')
        ax.set_ylabel('Inference Time (ms)')
        ax.set_xlabel('True Cardinality Range')

        # Prepare positioning for dodged boxes
        num_hues = len(hue_order)
        width = 0.8
        hue_width = width / max(num_hues, 1)

        # Remove seaborn's automatic legend (if present) to ensure single legend
        try:
            if hasattr(g, '_legend') and g._legend is not None:
                g._legend.remove()
        except Exception:
            pass

        # Add legend at the right side (single legend)
        from matplotlib.patches import Patch
        handles = [Patch(facecolor=palette.get(m, 'gray')) for m in hue_order]
        ax.legend(handles, hue_order, title='method', loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=28)

        # Add original rotated labels above each box (no numeric values)
        for i, bin_label in enumerate(labels):
            for j, method in enumerate(hue_order):
                subset = df[(df['bin'] == bin_label) & (df['method'] == method)]
                if subset.empty:
                    continue
                costs = subset['time']
                if costs.empty:
                    continue
                q1 = costs.quantile(0.25)
                q3 = costs.quantile(0.75)
                iqr = q3 - q1
                upper_lim = q3 + 1.5 * iqr
                whisker_val = costs[costs <= upper_lim].max()
                if pd.isna(whisker_val):
                    whisker_val = costs.max()
                x_pos = i + (j - num_hues/2 + 0.5) * hue_width
                color = palette.get(method, 'black')
                # original style: vertical, no bbox, no numeric value
                ax.text(x_pos, whisker_val * 1.25, method, rotation=90, ha='center', va='bottom', fontsize=28, color=color)

        ax.grid(True, which="both", ls="-", alpha=0.2)

        # Save figure
        g.fig.savefig(output_path, **SAVE_KWARGS)
        print(f"  -> Saved plot to {output_path}")
        plt.close(g.fig)

    except Exception as e:
        print(f"Error plotting: {e}")
        plt.close('all')


In [10]:
# Load FICE Times
try:
    if FICE_TIMES_PATH.exists():
        fice_all_data = pd.read_csv(FICE_TIMES_PATH)
        # Convert to ms (assuming source is in seconds/normalized units that need *1e3)
        fice_all_data['avg_inference_time'] = fice_all_data['avg_inference_time'] * 1e3
    else:
        print(f"Warning: {FICE_TIMES_PATH} not found. FICE data will be missing.")
        fice_all_data = pd.DataFrame()
except Exception as e:
    print(f"Error loading FICE times: {e}")
    fice_all_data = pd.DataFrame()

In [11]:

# List to collect all data for summary table
all_summary_data = []

for graph in GRAPHS:
    if graph not in CONFIG_MAP: continue
    gcare_prefix, fice_folder, fice_ds_name = CONFIG_MAP[graph]
    
    for q_type in ['star', 'path']:
        print(f"Processing {graph} - {q_type}...")
        
        combined_df = pd.DataFrame()

        # 1. GCARE Data
        gcare_subpath = f'{gcare_prefix}_{q_type}/{q_type}/{gcare_prefix}_{q_type}'
        gcare_full_path = GCARE_PATH / gcare_subpath
        
        if gcare_full_path.exists():
            try:
                gcare_df = helper.process_gcare_time(gcare_full_path)
                if not gcare_df.empty:
                    # Convert to ms (assuming source is seconds)
                    gcare_df['time'] = gcare_df['time'] * 1000
                    combined_df = pd.concat([combined_df, gcare_df], ignore_index=True)
            except Exception as e:
                print(f"  [Error] GCARE: {e}")
        else:
             print(f"  [Warning] GCARE path not found: {gcare_full_path}")

        # 2. FICE Data
        if not fice_all_data.empty:
             target_qt = f"{fice_ds_name}_{q_type}_queries"
             # Special cases
             if q_type == 'star':
                 alt_qt = f"{fice_ds_name}_queries"
                 if alt_qt in fice_all_data['query_type'].values:
                     target_qt = alt_qt
            
             fice_subset = fice_all_data[fice_all_data['query_type'] == target_qt].copy()
             
             if not fice_subset.empty:
                 fice_subset = fice_subset.rename(columns={'true_cardinality': 'true_value', 'avg_inference_time': 'time'})
                 fice_subset['method'] = 'fice' # Lowercase
                 fice_subset = fice_subset[['true_value', 'time', 'method']]
                 combined_df = pd.concat([combined_df, fice_subset], ignore_index=True)
             else:
                 print(f"  [Warning] No FICE data for {target_qt}")

        # Plot
        if not combined_df.empty:
            plot_time_boxplot_per_cardinality(combined_df, OUTPUT_DIR / f'boxplot_time_{graph}_{q_type}.pdf')
            
            # Collect data for summary
            # We want mean time per method for this dataset/query_type
            means = combined_df.groupby('method')['time'].mean().to_dict()
            means['graph'] = graph
            means['type'] = q_type
            all_summary_data.append(means)


Processing swdf - star...
  -> Saved plot to plots_output/boxplot_time_swdf_star.pdf
Processing swdf - path...
  -> Saved plot to plots_output/boxplot_time_swdf_path.pdf
Processing dbpedia100k - star...
  -> Saved plot to plots_output/boxplot_time_dbpedia100k_star.pdf
Processing dbpedia100k - path...
  -> Saved plot to plots_output/boxplot_time_dbpedia100k_path.pdf
Processing fb237_v4 - star...
  -> Saved plot to plots_output/boxplot_time_fb237_v4_star.pdf
Processing fb237_v4 - path...
  -> Saved plot to plots_output/boxplot_time_fb237_v4_path.pdf
Processing lubm - star...
  -> Saved plot to plots_output/boxplot_time_lubm_star.pdf
Processing lubm - path...
  -> Saved plot to plots_output/boxplot_time_lubm_path.pdf
Processing wikidata - star...
  -> Saved plot to plots_output/boxplot_time_wikidata_star.pdf
Processing wikidata - path...
  -> Saved plot to plots_output/boxplot_time_wikidata_path.pdf
Processing wn18rr_v4 - star...
  -> Saved plot to plots_output/boxplot_time_wn18rr_v4_star

In [12]:

# Create Summary DataFrame
summary_df = pd.DataFrame(all_summary_data)

if not summary_df.empty:
    # Reorder columns
    cols = ['graph', 'type'] + [c for c in summary_df.columns if c not in ['graph', 'type']]
    summary_df = summary_df[cols]
    
    # Calculate overall mean per method
    numeric_cols = summary_df.select_dtypes(include=[np.number]).columns
    overall_mean = summary_df[numeric_cols].mean()
    overall_row = pd.DataFrame(overall_mean).T
    overall_row['graph'] = 'OVERALL'
    overall_row['type'] = '-'
    
    # Append overall row
    final_summary = pd.concat([summary_df, overall_row], ignore_index=True)
    
    # Save to CSV
    csv_path = OUTPUT_DIR / 'runtime_summary_means.csv'
    final_summary.to_csv(csv_path, index=False)
    print(f"\nSummary table saved to {csv_path}")
    
    # Display
    display(final_summary)
else:
    print("No data available for summary table.")



Summary table saved to plots_output/runtime_summary_means.csv


,graph,type,cset,fice,wj
0,swdf,star,686.728941,0.227164,3098.248528
1,swdf,path,686.682775,0.228563,3097.958975
2,dbpedia100k,star,540.207964,0.239367,3128.750227
3,dbpedia100k,path,488.165786,0.239445,3100.091710
4,fb237_v4,star,107.498210,0.236826,3077.139553
5,fb237_v4,path,108.339580,0.234602,3078.459179
6,lubm,star,288.309649,0.239169,3177.234873
7,lubm,path,288.450259,0.236067,3177.594304
8,wikidata,star,1213.697120,0.233429,3545.671086
9,wikidata,path,1212.581689,0.233360,3545.049537
